<a href="https://colab.research.google.com/github/nazwapranoto/Sephora-Data-Analysis/blob/main/Sephora_Skincare_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Membaca dataset
df_product = pd.read_csv('product_info.csv')

# Menampilkan 5 baris pertama dari data produk
df_product.head()

FileNotFoundError: [Errno 2] No such file or directory: 'product_info.csv'

1. Analisis Harga vs Kepuasan: Apakah produk yang harganya lebih mahal selalu mendapatkan rating yang lebih tinggi dari pengguna?
2. Dominasi Brand: Brand apa saja yang merajai kategori skincare dengan rating tertinggi di Sephora?
3. Bedah Kandungan (The Ultimate Flex): Apa saja ingredients utama yang membedakan produk dengan rating tinggi dan rendah?

In [ ]:
# Mengecek informasi ringkas dari dataset (jumlah baris, nama kolom, dan tipe data)
print("--- INFO DATASET ---")
df_product.info()

print("\n--- CEK DATA KOSONG ---")

# Mengecek jumlah data yang bolong (Missing Values) di tiap kolom
print(df_product.isnull().sum())

In [ ]:
# 1. Memilih kolom-kolom yang relevan dengan pertanyaan bisnis
kolom_penting = ['product_name', 'brand_name', 'price_usd', 'rating', 'reviews', 'loves_count', 'ingredients', 'primary_category']
df_clean = df_product[kolom_penting].copy()

# 2. Membuang produk yang tidak memiliki data rating atau ingredients
df_clean = df_clean.dropna(subset=['rating', 'ingredients'])

# 3. Mereset index tabel agar berurutan kembali setelah ada baris yang dihapus
df_clean = df_clean.reset_index(drop=True)

# 4. Mengecek hasil akhir data kita yang sudah bersih
print("Sisa produk setelah dibersihkan:", df_clean.shape[0])
print("\nCek data kosong di dataset bersih:")
print(df_clean.isnull().sum())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Membuat scatter plot Harga vs Rating
sns.scatterplot(data=df_clean, x='price_usd', y='rating', alpha=0.4, color='teal')

# Menambahkan judul dan label sumbu
plt.title('Hubungan Antara Harga Produk (USD) dan Rating Pelanggan di Sephora', fontsize=14, fontweight='bold')
plt.xlabel('Harga (USD)', fontsize=12)
plt.ylabel('Rating (Skala 1-5)', fontsize=12)

plt.show()

Kesimpulan #1: Harga Mahal Bukan Jaminan Kepuasan Pelanggan
Berdasarkan analisis persebaran data Sephora, mayoritas produk kosmetik dan skincare berada di rentang harga di bawah $100 dan berhasil memperoleh rating sangat tinggi (di atas 4.0). Tidak ditemukan korelasi positif yang kuat antara tingginya harga produk dengan tingginya rating pelanggan.

In [ ]:
# 1. Memfilter hanya produk dengan rating sangat tinggi (Kualitas Premium)
high_rated = df_clean[df_clean['rating'] >= 4.5]

# 2. Menghitung jumlah produk berkualitas dari masing-masing brand, ambil Top 10
top_brands = high_rated['brand_name'].value_counts().head(10)

# 3. Membuat visualisasi Bar Chart
plt.figure(figsize=(12, 6))
sns.barplot(x=top_brands.values, y=top_brands.index, palette='magma')

plt.title('Top 10 Brand Sephora dengan Jumlah Produk Rating Tinggi (>= 4.5)', fontsize=14, fontweight='bold')
plt.xlabel('Jumlah Produk', fontsize=12)
plt.ylabel('Nama Brand', fontsize=12)

plt.show()

Kesimpulan #2: Dominasi In-House Brand dan Kekuatan Merek Luxury
Merek in-house (SEPHORA COLLECTION) berhasil memimpin jumlah produk dengan rating tertinggi, kemungkinan karena variasi katalog produknya yang sangat masif dan harga yang kompetitif. Namun, brand luxury konvensional (seperti Dior dan YSL) tetap menunjukkan dominasi kualitas yang sangat kuat dan mempertahankan loyalitas konsumen di pasar kosmetik.

In [ ]:
# 1. Daftar bahan aktif populer untuk dianalisis
active_ingredients = [
    'Niacinamide', 'Hyaluronic Acid', 'Salicylic Acid', 'Retinol',
    'Vitamin C', 'Ceramide', 'Glycerin', 'Panthenol', 'Peptide', 'Squalane'
]

# 2. Bikin tempat untuk menyimpan hasil hitungan
ingredient_counts = {}

# 3. Hanya ambil data bahan dari produk dengan rating tinggi (>= 4.5)
high_rated_ingredients = df_clean[df_clean['rating'] >= 4.5]['ingredients'].dropna()

# 4. Looping untuk menghitung kemunculan tiap bahan kimia di dalam daftar ingredients
for ingredient in active_ingredients:
    # Menghitung berapa kali kata tersebut muncul (mengabaikan huruf besar/kecil)
    count = high_rated_ingredients.str.contains(ingredient, case=False, na=False).sum()
    ingredient_counts[ingredient] = count

# 5. Mengubah hasil hitungan jadi tabel (DataFrame) biar gampang divisualisasikan
import pandas as pd
df_ing = pd.DataFrame(list(ingredient_counts.items()), columns=['Ingredient', 'Count'])
df_ing = df_ing.sort_values(by='Count', ascending=False)

# 6. Membuat visualisasi Bar Chart
plt.figure(figsize=(12, 6))
sns.barplot(x='Count', y='Ingredient', data=df_ing, palette='viridis')

# 7. Mempercantik grafik
plt.title('Top 10 Bahan Aktif pada Produk Sephora Berating Tinggi (>= 4.5)', fontsize=14, fontweight='bold')
plt.xlabel('Jumlah Produk yang Mengandung Bahan Tersebut', fontsize=12)
plt.ylabel('Bahan Aktif Kimia', fontsize=12)

plt.show()

Kesimpulan #3: Kunci Kepuasan Pelanggan Ada pada Hidrasi dan Skin Barrier
Bahan aktif yang paling mendominasi produk skincare berating tinggi bukanlah eksfoliator keras, melainkan agen hidrasi dan penenang kulit seperti Glycerin (humektan), Panthenol (Vitamin B5), dan Squalane. Hal ini menunjukkan bahwa konsumen Sephora sangat menghargai produk dengan formulasi dasar yang aman, melembapkan, dan tidak memicu iritasi (gentle formulation).